# OLYMPIC AI 2026 - TÁC VỤ 1: MEMD-ABSA (ACSTE)
## Quy trình sinh kết quả dự đoán với mô hình T5-Base (generate_result.ipynb)

Notebook này nạp checkpoint **T5-Base** đã được fine-tune chuyên biệt, thực thi suy luận trên tập dữ liệu kiểm tra và xuất ra file `submission.json` chuẩn định dạng của Ban Tổ Chức.

In [1]:
import os
import json
import re
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

ALL_DOMAINS = ["Restaurant", "Laptop", "Hotel", "Books", "Clothing"]
FIELD_SEP = " | "
TRIPLE_JOIN = " ; "
VALID_SENTIMENTS = {"POS", "NEG", "NEU"}

In [2]:
# Đường dẫn checkpoint và dữ liệu
def resolve_model_dir():
    cands = [
        Path("outputs/t5_base/checkpoint-best"),
        Path("checkpoint-best"),
        Path("models/t5-base"),
        Path("../outputs/t5_base/checkpoint-best"),
    ]
    for c in cands:
        if c.exists():
            return c
    return Path("models/t5-base")

model_path = resolve_model_dir()
print(f"Nạp mô hình từ: {model_path}")
tokenizer = AutoTokenizer.from_pretrained(str(model_path), local_files_only=True)
model = AutoModelForSeq2SeqLM.from_pretrained(str(model_path), local_files_only=True).to(device)
model.eval()
print("Nạp mô hình thành công!")

In [3]:
def parse_generated_text(text: str):
    if not text or text.strip().lower() in ("<empty>", "none", "n/a", "empty"):
        return []
    triples = []
    for chunk in re.split(r"\s*;\s*", text.strip()):
        parts = chunk.split(FIELD_SEP)
        if len(parts) != 3:
            parts = [p.strip() for p in chunk.split("|")]
        if len(parts) == 3:
            a, c, s = (p.strip() for p in parts)
            s_clean = s.upper()
            if s_clean not in VALID_SENTIMENTS:
                s_clean = "POS" if "POS" in s_clean else ("NEG" if "NEG" in s_clean else "NEU")
            triples.append({"aspect": a if a.upper() != "NULL" else "NULL", "category": c, "sentiment": s_clean})
    return triples

def post_process_triples(triples, raw_words):
    seen = set()
    res = []
    raw_lower = raw_words.lower()
    for t in triples:
        a, c, s = t["aspect"], t["category"], t["sentiment"]
        if a != "NULL" and a not in raw_words:
            idx = raw_lower.find(a.lower())
            if idx != -1:
                a = raw_words[idx : idx + len(a)]
            else:
                cl = a.strip(".,!?:;\"'()[]{}")
                idx2 = raw_lower.find(cl.lower())
                if idx2 != -1:
                    a = raw_words[idx2 : idx2 + len(cl)]
                else:
                    a = "NULL"
        key = (a, c, s)
        if key not in seen:
            seen.add(key)
            res.append({"aspect": a, "category": c, "sentiment": s})
    return res

In [4]:
# Dự đoán trên tập dữ liệu kiểm tra
def predict_dataset(test_file_path: str, output_submission_path: str = "submission.json"):
    with open(test_file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    submission = {}
    for domain in ALL_DOMAINS:
        items = data.get(domain, [])
        domain_preds = []
        batch_size = 8
        for i in range(0, len(items), batch_size):
            b_items = items[i : i + batch_size]
            prompts = [f"acste [{domain}]: {x["raw_words"].strip()}" for x in b_items]
            enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
            with torch.no_grad():
                gens = model.generate(**enc, max_new_tokens=192, num_beams=2, early_stopping=True)
            dec = tokenizer.batch_decode(gens, skip_special_tokens=True)
            for item, text in zip(b_items, dec):
                raw_t = parse_generated_text(text)
                clean_t = post_process_triples(raw_t, item["raw_words"])
                domain_preds.append({"raw_words": item["raw_words"], "triples": clean_t})
        submission[domain] = domain_preds
        print(f"Hoàn thành domain {domain}: {len(domain_preds)} câu.")
    with open(output_submission_path, "w", encoding="utf-8") as f:
        json.dump(submission, f, ensure_ascii=False, indent=2)
    print(f"Đã lưu submission tại: {output_submission_path}")

test_path = "data/TACVU1/private_test/PrivateTest.json"
if Path(test_path).exists():
    predict_dataset(test_path, "submission.json")